In [1]:
import pandas as pd
print(pd.__version__)

3.0.3


In [4]:
import pandas as pd

df = pd.read_csv("../data/raw/german_credit_data.csv")
df.head()

,Creditability,Account Balance,Duration of Credit (month),Payment Status of Previous Credit,Purpose,Credit Amount,Value Savings/Stocks,Length of current employment,Instalment per cent,Sex & Marital Status,...,Duration in Current address,Most valuable available asset,Age (years),Concurrent Credits,Type of apartment,No of Credits at this Bank,Occupation,No of dependents,Telephone,Foreign Worker
0,1,1,18,4,2,1049,1,2,4,2,...,4,2,21,3,1,1,3,1,1,1
1,1,1,9,4,0,2799,1,3,2,3,...,2,1,36,3,1,2,3,2,1,1
2,1,2,12,2,9,841,2,4,2,2,...,4,1,23,3,1,1,2,1,1,1
3,1,1,12,4,0,2122,1,3,3,3,...,2,1,39,3,1,2,2,2,1,2
4,1,1,12,4,0,2171,1,3,4,3,...,4,2,38,1,2,2,2,1,1,2


In [6]:
df = pd.read_csv("../data/raw/german_credit_data.csv")
print(df.shape)
df.head()

(1000, 21)


,Creditability,Account Balance,Duration of Credit (month),Payment Status of Previous Credit,Purpose,Credit Amount,Value Savings/Stocks,Length of current employment,Instalment per cent,Sex & Marital Status,...,Duration in Current address,Most valuable available asset,Age (years),Concurrent Credits,Type of apartment,No of Credits at this Bank,Occupation,No of dependents,Telephone,Foreign Worker
0,1,1,18,4,2,1049,1,2,4,2,...,4,2,21,3,1,1,3,1,1,1
1,1,1,9,4,0,2799,1,3,2,3,...,2,1,36,3,1,2,3,2,1,1
2,1,2,12,2,9,841,2,4,2,2,...,4,1,23,3,1,1,2,1,1,1
3,1,1,12,4,0,2122,1,3,3,3,...,2,1,39,3,1,2,2,2,1,2
4,1,1,12,4,0,2171,1,3,4,3,...,4,2,38,1,2,2,2,1,1,2


In [10]:
df['Creditability'].value_counts()
df['Creditability'].value_counts(normalize=True)

Creditability
1    0.7
0    0.3
Name: proportion, dtype: float64

In [11]:
df.isnull().sum()

Creditability                        0
Account Balance                      0
Duration of Credit (month)           0
Payment Status of Previous Credit    0
Purpose                              0
Credit Amount                        0
Value Savings/Stocks                 0
Length of current employment         0
Instalment per cent                  0
Sex & Marital Status                 0
Guarantors                           0
Duration in Current address          0
Most valuable available asset        0
Age (years)                          0
Concurrent Credits                   0
Type of apartment                    0
No of Credits at this Bank           0
Occupation                           0
No of dependents                     0
Telephone                            0
Foreign Worker                       0
dtype: int64

In [15]:
df.groupby('Account Balance')['Creditability'].mean()


Account Balance
1    0.507299
2    0.609665
3    0.777778
4    0.883249
Name: Creditability, dtype: float64

In [16]:
df['Account Balance'].value_counts()

Account Balance
4    394
1    274
2    269
3     63
Name: count, dtype: int64

In [17]:
print(df.groupby('Payment Status of Previous Credit')['Creditability'].mean())
print(df['Payment Status of Previous Credit'].value_counts())

Payment Status of Previous Credit
0    0.375000
1    0.428571
2    0.681132
3    0.681818
4    0.829352
Name: Creditability, dtype: float64
Payment Status of Previous Credit
2    530
4    293
3     88
1     49
0     40
Name: count, dtype: int64


In [18]:
df['Duration_bin'] = pd.cut(df['Duration of Credit (month)'], bins=5)
print(df.groupby('Duration_bin')['Creditability'].mean())
print(df['Duration_bin'].value_counts())

Duration_bin
(3.932, 17.6]    0.792148
(17.6, 31.2]     0.677665
(31.2, 44.8]     0.582524
(44.8, 58.4]     0.410714
(58.4, 72.0]     0.500000
Name: Creditability, dtype: float64
Duration_bin
(3.932, 17.6]    433
(17.6, 31.2]     394
(31.2, 44.8]     103
(44.8, 58.4]      56
(58.4, 72.0]      14
Name: count, dtype: int64


In [19]:
df['CreditAmount_bin'] = pd.cut(df['Credit Amount'], bins=5)
print(df.groupby('CreditAmount_bin')['Creditability'].mean())
print(df['CreditAmount_bin'].value_counts())

CreditAmount_bin
(231.826, 3884.8]     0.743902
(3884.8, 7519.6]      0.615819
(7519.6, 11154.4]     0.596491
(11154.4, 14789.2]    0.272727
(14789.2, 18424.0]    0.333333
Name: Creditability, dtype: float64
CreditAmount_bin
(231.826, 3884.8]     738
(3884.8, 7519.6]      177
(7519.6, 11154.4]      57
(11154.4, 14789.2]     22
(14789.2, 18424.0]      6
Name: count, dtype: int64


## EDA Summary — German Credit Dataset (Phase 1)

**Dataset overview**
- Shape: 1000 rows × 21 columns
- Source: UCI Statlog German Credit Data (numeric-encoded version)
- No missing values across any column
- Target column: `Creditability` (1 = Good credit, 0 = Bad credit)

**Class balance**
- 70% Good (1), 30% Bad (0) — moderate imbalance
- Plain accuracy is not a reliable metric; will use AUC-ROC, precision, recall, F1
- Imbalance handling (class_weight or SMOTE) to be decided explicitly in `train.py`

**Key predictors identified so far (all directionally consistent with domain intuition)**

| Feature | Trend | Strength |
|---|---|---|
| Account Balance | Higher balance → higher good-credit rate (51% → 88%) | Strong |
| Payment Status of Previous Credit | Better past payment history → higher good-credit rate (37% → 83%) | Strong, monotonic |
| Duration of Credit (month) | Longer duration → lower good-credit rate (79% → 41%) | Strong |
| Credit Amount | Larger loan amount → lower good-credit rate (74% → ~30%) | Strong |

**Caveats**
- Tail bins for Duration and Credit Amount (very long duration, very high amount) have small sample sizes (as low as 6–14 rows) — directional trend is trustworthy, exact percentages in those bins are not reliable and should be caveated if referenced later.
- Categorical columns (e.g. Account Balance, Payment Status, Purpose) are numerically encoded but are NOT ordinal in a strict mathematical sense — must be treated as categorical (not raw int) during feature engineering, per `column_mapping.md`.

**Next step:** Build `data_loader.py` and `features.py` in `src/`, migrating this cleaning/encoding logic out of the notebook per project architecture (Section 3).

## Phase 2 EDA — Lending Club Sample

In [20]:
df_lc = pd.read_csv("../data/raw/lending_club_sample.csv", low_memory=False)
print(df_lc.shape)
df_lc['loan_status'].value_counts()

(113035, 151)


loan_status
Fully Paid                                             54058
Current                                                43839
Charged Off                                            13254
Late (31-120 days)                                      1081
In Grace Period                                          446
Late (16-30 days)                                        198
Does not meet the credit policy. Status:Fully Paid       113
Does not meet the credit policy. Status:Charged Off       42
Default                                                    3
Name: count, dtype: int64

In [21]:
# Statuses that represent a resolved, usable outcome
good_statuses = ['Fully Paid', 'Does not meet the credit policy. Status:Fully Paid']
bad_statuses = ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off']

# Filter to only resolved loans
df_lc_filtered = df_lc[df_lc['loan_status'].isin(good_statuses + bad_statuses)].copy()

# Build binary target: 1 = good, 0 = bad (same convention as German Credit)
df_lc_filtered['target'] = df_lc_filtered['loan_status'].apply(
    lambda x: 1 if x in good_statuses else 0
)

print(df_lc_filtered.shape)
print(df_lc_filtered['target'].value_counts())
print(df_lc_filtered['target'].value_counts(normalize=True))

(67470, 152)
target
1    54171
0    13299
Name: count, dtype: int64
target
1    0.80289
0    0.19711
Name: proportion, dtype: float64


In [ ]:
missing_pct = df_lc_filtered.isnull().mean().sort_values(ascending=False) * 100
print(missing_pct.head(30))

member_id                                     100.000000
next_pymnt_d                                   99.765822
orig_projected_additional_accrued_interest     99.730250
hardship_payoff_balance_amount                 99.571661
hardship_reason                                99.571661
hardship_type                                  99.571661
hardship_amount                                99.571661
hardship_start_date                            99.571661
hardship_end_date                              99.571661
payment_plan_start_date                        99.571661
hardship_length                                99.571661
hardship_dpd                                   99.571661
hardship_loan_status                           99.571661
hardship_status                                99.571661
hardship_last_payment_amount                   99.571661
deferral_term                                  99.571661
sec_app_mths_since_last_major_derog            99.503483
sec_app_revol_util             

In [23]:
threshold = 50  # percent missing
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns with >{threshold}% missing")

df_lc_reduced = df_lc_filtered.drop(columns=cols_to_drop)
print(df_lc_reduced.shape)

Dropping 58 columns with >50% missing
(67470, 94)


In [24]:
leakage_cols = [
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d',
    'out_prncp', 'out_prncp_inv', 'debt_settlement_flag',
    'settlement_status', 'settlement_date', 'settlement_amount',
    'settlement_percentage', 'settlement_term',
    'issue_d',  # loan issue date - only known after approval, not at application
]

# Only drop ones actually present (some may have already been dropped by missing-value filter)
leakage_cols_present = [c for c in leakage_cols if c in df_lc_reduced.columns]
print(f"Dropping {len(leakage_cols_present)} known leakage columns:")
print(leakage_cols_present)

df_lc_clean = df_lc_reduced.drop(columns=leakage_cols_present)
print(df_lc_clean.shape)

Dropping 14 known leakage columns:
['total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d', 'out_prncp', 'out_prncp_inv', 'debt_settlement_flag', 'issue_d']
(67470, 80)


In [25]:
drop_further = ['grade', 'sub_grade', 'id', 'url', 'emp_title']

drop_further_present = [c for c in drop_further if c in df_lc_clean.columns]
print(f"Dropping {len(drop_further_present)} columns:")
print(drop_further_present)

df_lc_clean2 = df_lc_clean.drop(columns=drop_further_present)
print(df_lc_clean2.shape)

Dropping 5 columns:
['grade', 'sub_grade', 'id', 'url', 'emp_title']
(67470, 75)


In [26]:
print(df_lc_clean2.columns.tolist())

['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'pymnt_plan', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'policy_code', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl

In [27]:
# Check if policy_code is actually constant (before deciding to drop it)
print(df_lc_clean2['policy_code'].value_counts())

policy_code
1.0    67470
Name: count, dtype: int64


In [28]:
drop_final = [
    'funded_amnt', 'funded_amnt_inv', 'loan_status', 'title',
    'last_fico_range_high', 'last_fico_range_low', 'hardship_flag'
]

drop_final_present = [c for c in drop_final if c in df_lc_clean2.columns]
df_lc_final = df_lc_clean2.drop(columns=drop_final_present)
print(f"Dropped {len(drop_final_present)} more columns")
print(df_lc_final.shape)

Dropped 7 more columns
(67470, 68)


In [29]:
df_lc_final = df_lc_final.drop(columns=['policy_code'])
print(df_lc_final.shape)

(67470, 67)


In [30]:
remaining_missing = df_lc_final.isnull().mean().sort_values(ascending=False) * 100
print(remaining_missing[remaining_missing > 0])

mths_since_recent_inq         12.977620
num_tl_120dpd_2m               8.955091
mo_sin_old_il_acct             8.117682
emp_length                     5.777383
pct_tl_nvr_dlq                 5.264562
avg_cur_bal                    5.248258
num_op_rev_tl                  5.245294
mo_sin_old_rev_tl_op           5.245294
tot_cur_bal                    5.245294
total_rev_hi_lim               5.245294
num_tl_30dpd                   5.245294
num_rev_tl_bal_gt_0            5.245294
num_tl_90g_dpd_24m             5.245294
num_tl_op_past_12m             5.245294
mo_sin_rcnt_tl                 5.245294
mo_sin_rcnt_rev_tl_op          5.245294
num_rev_accts                  5.245294
num_accts_ever_120_pd          5.245294
num_actv_bc_tl                 5.245294
tot_hi_cred_lim                5.245294
num_actv_rev_tl                5.245294
num_bc_tl                      5.245294
total_il_high_credit_limit     5.245294
num_il_tl                      5.245294
tot_coll_amt                   5.245294
